In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# Step11.66 Reload immune scores and survival data
# ============================================================

import os
import pandas as pd
import numpy as np

project_dir = _hcc_path('')

step11_dir = os.path.join(
    project_dir,
    "LIHC_RNA_Step11_Immune"
)

result_dir = os.path.join(
    step11_dir,
    "Results"
)

input_dir = os.path.join(
    step11_dir,
    "Input"
)

fig_dir = os.path.join(
    step11_dir,
    "Figures"
)


# ssGSEA immune scores
ssgsea_matrix = pd.read_csv(
    os.path.join(
        result_dir,
        "LIHC_ssGSEA_Immune_Cell_Scores_365.csv"
    ),
    index_col=0
)


# Risk + survival
risk_immune = pd.read_csv(
    os.path.join(
        input_dir,
        "LIHC_RiskGroup_365.csv"
    ),
    index_col=0
)


print("ssGSEA:", ssgsea_matrix.shape)
print("Risk/survival:", risk_immune.shape)

print(ssgsea_matrix.columns.tolist())
print(risk_immune.columns.tolist())

In [ ]:
# ============================================================
# Step11.67 Merge immune scores with survival
# ============================================================

ssgsea_matrix.index.name = "Sample_ID"
risk_immune.index.name = "Sample_ID"


immune_surv = (
    ssgsea_matrix
    .reset_index()
    .merge(
        risk_immune.reset_index(),
        on="Sample_ID",
        how="inner"
    )
)


print("Merged shape:", immune_surv.shape)

print(
    immune_surv[
        ["OS_time", "OS"]
    ].isna().sum()
)

display(
    immune_surv.head()
)

In [ ]:
# ============================================================
# Step11.68 Continuous Cox analysis
# ============================================================

from lifelines import CoxPHFitter
from statsmodels.stats.multitest import multipletests


focus_cells = [
    "CD8_T_cells",
    "Cytotoxic_cells",
    "NK_cells",
    "Neutrophils",
    "B_cells"
]


immune_cox_results = []


for cell in focus_cells:

    temp = immune_surv[
        [
            "OS_time",
            "OS",
            cell
        ]
    ].dropna().copy()


    cph = CoxPHFitter()

    cph.fit(
        temp,
        duration_col="OS_time",
        event_col="OS"
    )


    row = cph.summary.loc[cell]


    immune_cox_results.append({
        "Cell_type": cell,
        "N": len(temp),
        "HR": np.exp(row["coef"]),
        "Lower_95CI": np.exp(row["coef lower 95%"]),
        "Upper_95CI": np.exp(row["coef upper 95%"]),
        "P_value": row["p"]
    })


immune_cox_results = pd.DataFrame(
    immune_cox_results
)


immune_cox_results["FDR"] = multipletests(
    immune_cox_results["P_value"],
    method="fdr_bh"
)[1]


immune_cox_results = (
    immune_cox_results
    .sort_values(["FDR", "P_value"])
    .reset_index(drop=True)
)


display(
    immune_cox_results
)

In [ ]:
# ============================================================
# Step11.69 Z-score standardized Cox
# HR per 1-SD increase in immune score
# ============================================================

immune_cox_z_results = []


for cell in focus_cells:

    temp = immune_surv[
        [
            "OS_time",
            "OS",
            cell
        ]
    ].dropna().copy()


    mean_val = temp[cell].mean()
    std_val = temp[cell].std()


    temp[cell + "_Z"] = (
        temp[cell] - mean_val
    ) / std_val


    cph = CoxPHFitter()

    cph.fit(
        temp[
            [
                "OS_time",
                "OS",
                cell + "_Z"
            ]
        ],
        duration_col="OS_time",
        event_col="OS"
    )


    row = cph.summary.loc[
        cell + "_Z"
    ]


    immune_cox_z_results.append({
        "Cell_type": cell,
        "N": len(temp),
        "HR_per_1SD": np.exp(row["coef"]),
        "Lower_95CI": np.exp(row["coef lower 95%"]),
        "Upper_95CI": np.exp(row["coef upper 95%"]),
        "P_value": row["p"]
    })


immune_cox_z_results = pd.DataFrame(
    immune_cox_z_results
)


immune_cox_z_results["FDR"] = multipletests(
    immune_cox_z_results["P_value"],
    method="fdr_bh"
)[1]


immune_cox_z_results = (
    immune_cox_z_results
    .sort_values(
        ["FDR", "P_value"]
    )
    .reset_index(drop=True)
)


display(
    immune_cox_z_results
)

In [ ]:
immune_cox_results.to_csv(
    os.path.join(
        result_dir,
        "LIHC_ImmuneCell_OS_Cox_RawScore.csv"
    ),
    index=False
)


immune_cox_z_results.to_csv(
    os.path.join(
        result_dir,
        "LIHC_ImmuneCell_OS_Cox_Zscore.csv"
    ),
    index=False
)


print("Immune-cell Cox results saved.")

In [ ]:
# ============================================================
# Step11.71 Prepare KM groups
# ============================================================

for cell in focus_cells:

    cutoff = immune_surv[
        cell
    ].median()

    immune_surv[
        cell + "_Group"
    ] = np.where(
        immune_surv[cell] > cutoff,
        "High",
        "Low"
    )


for cell in focus_cells:

    print("\n", cell)

    print(
        immune_surv[
            cell + "_Group"
        ].value_counts()
    )

In [ ]:
# ============================================================
# Step11.72
# KM curves for key immune-cell signatures
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines import CoxPHFitter


km_cells = [
    "CD8_T_cells",
    "Cytotoxic_cells",
    "NK_cells",
    "Neutrophils",
    "B_cells"
]


fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 9)
)

axes = axes.flatten()


for ax, cell in zip(
    axes,
    km_cells
):

    temp = immune_surv[
        [
            "OS_time",
            "OS",
            cell
        ]
    ].dropna().copy()

    cutoff = temp[cell].median()

    temp["Group"] = np.where(
        temp[cell] > cutoff,
        "High",
        "Low"
    )


    high = temp[
        temp["Group"] == "High"
    ]

    low = temp[
        temp["Group"] == "Low"
    ]


    # KM
    kmf_high = KaplanMeierFitter()
    kmf_low = KaplanMeierFitter()


    kmf_high.fit(
        high["OS_time"],
        event_observed=high["OS"],
        label="High"
    )

    kmf_low.fit(
        low["OS_time"],
        event_observed=low["OS"],
        label="Low"
    )


    kmf_high.plot_survival_function(
        ax=ax,
        ci_show=False
    )

    kmf_low.plot_survival_function(
        ax=ax,
        ci_show=False
    )


    # Log-rank
    lr = logrank_test(
        high["OS_time"],
        low["OS_time"],
        event_observed_A=high["OS"],
        event_observed_B=low["OS"]
    )


    # Dichotomized Cox HR: High vs Low
    temp["Group_code"] = np.where(
        temp["Group"] == "High",
        1,
        0
    )

    cph = CoxPHFitter()

    cph.fit(
        temp[
            [
                "OS_time",
                "OS",
                "Group_code"
            ]
        ],
        duration_col="OS_time",
        event_col="OS"
    )

    row = cph.summary.loc[
        "Group_code"
    ]

    hr = np.exp(
        row["coef"]
    )

    lower = np.exp(
        row["coef lower 95%"]
    )

    upper = np.exp(
        row["coef upper 95%"]
    )


    if lr.p_value < 0.001:
        ptxt = "P < 0.001"
    else:
        ptxt = f"P = {lr.p_value:.3f}"


    ax.text(
        0.05,
        0.16,
        (
            f"{ptxt}\n"
            f"HR = {hr:.2f}\n"
            f"95% CI: {lower:.2f}–{upper:.2f}"
        ),
        transform=ax.transAxes,
        fontsize=10
    )


    ax.set_title(
        cell.replace("_", " "),
        fontsize=12
    )

    ax.set_xlabel(
        "Time (days)"
    )

    ax.set_ylabel(
        "Overall survival probability"
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


# 删除最后空白图
for j in range(
    len(km_cells),
    len(axes)
):
    fig.delaxes(
        axes[j]
    )


fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneCell_OS_KM_Combined.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneCell_OS_KM_Combined.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11.73
# Forest plot of immune-cell Cox results
# ============================================================

forest_df = immune_cox_z_results.copy()

forest_df = forest_df.sort_values(
    "HR_per_1SD"
).reset_index(drop=True)


y = np.arange(
    len(forest_df)
)


fig, ax = plt.subplots(
    figsize=(7.5, 5.5)
)


ax.errorbar(
    forest_df["HR_per_1SD"],
    y,
    xerr=[
        forest_df["HR_per_1SD"] - forest_df["Lower_95CI"],
        forest_df["Upper_95CI"] - forest_df["HR_per_1SD"]
    ],
    fmt="o",
    capsize=4
)


ax.axvline(
    1,
    linestyle="--",
    linewidth=1
)


ax.set_yticks(y)

ax.set_yticklabels(
    forest_df["Cell_type"]
    .str.replace("_", " ")
)


ax.set_xlabel(
    "Hazard ratio per 1-SD increase"
)

ax.set_title(
    "Association between immune-cell infiltration and overall survival"
)


# 标注 HR 和 FDR
for i, row in forest_df.iterrows():

    text = (
        f'HR {row["HR_per_1SD"]:.2f} '
        f'({row["Lower_95CI"]:.2f}–{row["Upper_95CI"]:.2f})'
    )

    ax.text(
        row["Upper_95CI"] + 0.03,
        i,
        text,
        va="center",
        fontsize=9
    )


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneCell_OS_Cox_Forest.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneCell_OS_Cox_Forest.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)